# 09 - Test-Time Compute Scaling (Phase 5, redesigned)

**Mode:** GPU (T4 x2 session, GPU 0), background commit, 2 h hard cap. Estimated ~1.0 GPU hour.

## Question
If we let the model try more times at inference, how much does accuracy improve, and how much of that can we realise **without an oracle**?

## Two curves from ONE batch of n=16 samples per task
| curve | meaning |
|---|---|
| **Oracle Pass@k** | unbiased estimate that at least one of k samples passes all tests (needs the hidden tests to know which one) |
| **Visible-test selection** | the model sees one example test in its prompt. Draw k samples, keep those that pass that example, pick one at random, score it on the full tests. This is deployable: no hidden information |

Computing K = 1, 2, 4, 8, 16 from a single set of 16 samples is ~5x cheaper than five separate runs (the local Phase 5 ran each K separately).

The gap between the two curves is the cost of not having an oracle; samples that pass the visible example but fail hidden tests are the "false friends".

In [ ]:
INSTALL_WHEELS = False

import glob, os, subprocess, sys, time
T0 = time.time()

def find_input(name):
    hits = [p for p in glob.glob(f"/kaggle/input/**/{name}", recursive=True) if os.path.isdir(p)]
    assert hits, f"dataset '{name}' not attached; found: {os.listdir('/kaggle/input')}"
    return hits[0]

UTILS, DATA = find_input("llm-lab-utils"), find_input("llm-lab-data")
WHEELS = find_input("llm-lab-wheels") if INSTALL_WHEELS else None

# Kaggle's image ships torchao 0.10, which peft>=0.20 refuses to import. We do not use it.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], capture_output=True)

if INSTALL_WHEELS:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps",
                        *sorted(glob.glob(f"{WHEELS}/*.whl"))], capture_output=True, text=True)
    print("offline pip install rc:", r.returncode, r.stderr[-500:])

if os.path.exists(f"{UTILS}/__init__.py"):
    os.makedirs("/tmp/_pkg", exist_ok=True)
    if not os.path.exists("/tmp/_pkg/llm_lab_utils"):
        os.symlink(UTILS, "/tmp/_pkg/llm_lab_utils")
    sys.path.insert(0, "/tmp/_pkg")
else:
    found = glob.glob(f"{UTILS}/**/llm_lab_utils/__init__.py", recursive=True)
    assert found, "llm_lab_utils package not found in dataset"
    sys.path.insert(0, os.path.dirname(os.path.dirname(found[0])))
import llm_lab_utils
print("llm_lab_utils", llm_lab_utils.__version__, "from", llm_lab_utils.__file__)

In [ ]:
import json, gc, torch, psutil, pandas as pd
from concurrent.futures import ThreadPoolExecutor
from llm_lab_utils import (PRIMARY_MODEL_ID, SubprocessVerifier, ExperimentTracker, load_dataset, pass_at_k)
from llm_lab_utils.model_utils import setup_model_and_tokenizer
from llm_lab_utils.evaluator import run_evaluation
from llm_lab_utils.metrics import selection_accuracy

MODEL_CHOICE = "base"      # <- change after NB 07: base | sft | grpo_from_base | grpo_from_sft
def one(pattern):
    hits = glob.glob(pattern, recursive=True); assert hits, f"not found: {pattern}"; return hits[0]
ADAPTERS = {"base": None,
            "sft": lambda: [one(f"{find_input('04-sft-training')}/**/exp01_sft_mbpp_v1/checkpoint-final")],
            "grpo_from_base": lambda: [one(f"{find_input('06-grpo-training')}/**/grpo_from_base/checkpoint-final")],
            "grpo_from_sft": lambda: [one(f"{find_input('04-sft-training')}/**/exp01_sft_mbpp_v1/checkpoint-final"),
                                      one(f"{find_input('06-grpo-training')}/**/grpo_from_sft/checkpoint-final")]}
adapters = ADAPTERS[MODEL_CHOICE]() if ADAPTERS[MODEL_CHOICE] else None
print("model:", MODEL_CHOICE, adapters)
model, tokenizer = setup_model_and_tokenizer(PRIMARY_MODEL_ID, dtype="float16", device_map={"": 0}, adapter_path=adapters, merge=True)
model.eval()

GEN_CFG = {"max_new_tokens": 384, "temperature": 0.8, "top_p": 0.95, "do_sample": True}   # = local search_complex.yaml (higher T for diversity)
N, KS, SEED = 16, [1, 2, 4, 8, 16], 1234
tasks = load_dataset(f"{DATA}/mbpp_v2_eval.jsonl")
verifier = SubprocessVerifier(timeout=5.0, max_memory_mb=512)

## 1. Sample 16 completions per task and verify on the full tests

In [ ]:
EXP_ID = f"exp04_scaling_{MODEL_CHOICE}"
t0 = time.time(); torch.cuda.reset_peak_memory_stats(0)
tracker = ExperimentTracker(EXP_ID, "/kaggle/working")
tracker.set_config(model=f"{PRIMARY_MODEL_ID} ({MODEL_CHOICE}: {adapters})", dataset="mbpp_v2_eval.jsonl",
                   hyperparameters={**GEN_CFG, "n": N, "k_values": KS, "seed": SEED, "dtype": "float16", "merged": True})
metrics, task_res = run_evaluation(model, tokenizer, tasks, verifier, GEN_CFG, KS, tracker=tracker, seed=SEED, sample_chunk=16)
gen_hours = (time.time() - t0) / 3600
print({k: round(v, 4) for k, v in metrics.items()}, f"| {gen_hours:.2f} h")

## 2. Re-verify every sample against the VISIBLE example test only

In [ ]:
gens = [json.loads(l) for l in open(f"/kaggle/working/{EXP_ID}/generations.jsonl")]
by_id = {t.id: t.model_dump() for t in tasks}

def visible_only(g):
    td = dict(by_id[g["task_id"]]); td["hidden_tests"] = []
    return g["task_id"], bool(verifier.verify(td, g["candidate_code"])["passed"]), bool(g["verification"]["passed"])

with ThreadPoolExecutor(max_workers=8) as ex:
    rows = list(ex.map(visible_only, gens))
per_task = {}
for tid, vis, full in rows: per_task.setdefault(tid, []).append({"visible": vis, "full": full})
assert all(len(v) == N for v in per_task.values()), {len(v) for v in per_task.values()}
n_vis = sum(s["visible"] for v in per_task.values() for s in v); n_false = sum(s["visible"] and not s["full"] for v in per_task.values() for s in v)
print(f"samples passing the visible example: {n_vis}/{len(rows)};  of those, failing hidden tests ('false friends'): {n_false} ({n_false / max(n_vis, 1):.1%})")

## 3. Scaling curves

In [ ]:
oracle = {k: sum(pass_at_k(N, sum(s["full"] for s in v), k) for v in per_task.values()) / len(per_task) for k in KS}
select = {k: sum(selection_accuracy(v, k, seed=7) for v in per_task.values()) / len(per_task) for k in KS}
curve = pd.DataFrame({"oracle_pass@k": oracle, "visible_test_selection": select}).round(3)
curve["gap"] = (curve["oracle_pass@k"] - curve["visible_test_selection"]).round(3)
curve["extra_samples_vs_k1"] = [k for k in KS]
print(curve.to_string())

import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
plt.figure(figsize=(6.5, 4)); plt.plot(KS, list(oracle.values()), "o-", label="oracle Pass@k")
plt.plot(KS, list(select.values()), "s-", label="visible-test selection"); plt.xscale("log", base=2); plt.xticks(KS, KS)
plt.xlabel("samples per task (k)"); plt.ylabel("accuracy on held-out MBPP"); plt.title(f"Test-time scaling ({MODEL_CHOICE})")
plt.legend(); plt.grid(alpha=.3); plt.savefig(f"/kaggle/working/{EXP_ID}/scaling_curve.png", dpi=120)

## 4. Save

In [ ]:
tracker.update_metrics({"oracle_pass_at_k": oracle, "visible_selection": select, "false_friend_rate": n_false / max(n_vis, 1)})
wall_total = (time.time() - T0) / 3600
tracker.update_hardware(torch.cuda.max_memory_allocated(0) / 1e9, psutil.virtual_memory().used / 1e9)
tracker.data["training"].update(wall_hours=round(wall_total, 4), gpu_hours=round(wall_total, 4)); tracker.save()
print(f"saved. wall = GPU hours = {wall_total:.2f}")

## Takeaways (fill in)
- How much does accuracy grow from k=1 to k=16, and does it flatten (diminishing returns per GPU-hour)?
- How much of the oracle gain survives with only the visible example as a filter? The gap is the price of a deployable system.
- Compare with RL: if GRPO's Pass@1 (NB 07) reaches the visible-selection accuracy at k=4 or 8, training moved compute from inference to the weights.